# 🛰️ SatQuery AI: Vision-Language LoRA Training & Export (Steps 3, 4 & 5)
### Fine-Tuning Qwen2-VL-2B for Remote Sensing on Free Google Colab / Kaggle T4 GPUs

**Roadmap Covered in this Notebook:**
1. **Step 3: Parameter-Efficient Fine-Tuning (QLoRA)** on Multi-Task Remote Sensing Data
2. **Step 4: Model Export & Adapter Serialization** (`.safetensors`)
3. **Step 5: Production Inference Testing** with `run_specialist_model`

> **GPU Note:** Go to `Runtime` -> `Change runtime type` -> select `T4 GPU` (Free Tier) or `A100`.

In [ ]:
# [Cell 1] Install Training Dependencies
!pip install -q --upgrade pip
!pip install -q "transformers>=4.45.0" "peft>=0.12.0" "accelerate>=0.34.0" "datasets>=2.21.0" "bitsandbytes>=0.43.0" "qwen-vl-utils>=0.0.8" pillow matplotlib

In [ ]:
# [Cell 2] GPU & VRAM Verification
import torch
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🚀 GPU detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    print(f"⚡ BF16 Supported: {torch.cuda.is_bf16_supported()}")
else:
    print("⚠️ GPU not detected! Switch to GPU runtime in Colab menu.")
print("=" * 60)

In [ ]:
# [Cell 3] Load Base Qwen2-VL-2B-Instruct with 4-bit QLoRA
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print(f"Loading processor and base model: {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=1024*28*28)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

# Prepare model for gradient checkpointing & LoRA
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# [Cell 4] Load Dataset & Setup PyTorch Collator
from datasets import DatasetDict
from qwen_vl_utils import process_vision_info

# Load preprocessed dataset from Step 2
try:
    dataset = DatasetDict.load_from_disk("./satquery_data/processed")
    train_ds = dataset["train"]
    val_ds = dataset["validation"]
    print(f"Loaded dataset: {len(train_ds)} train samples, {len(val_ds)} val samples.")
except Exception as e:
    print(f"Local dataset not found, generating sample dataset: {e}")
    # Fallback to in-memory synthetic data
    from src.data.downloaders import generate_synthetic_satellite_dataset
    from src.data.formatters import SatQueryDatasetFormatter
    samples = generate_synthetic_satellite_dataset(num_samples_per_task=20)
    dataset = SatQueryDatasetFormatter().build_hf_dataset_dict(samples)
    train_ds = dataset["train"]
    val_ds = dataset["validation"]

class ColabDataCollator:
    def __init__(self, proc):
        self.processor = proc
    def __call__(self, batch):
        convs = [item["conversations"] for item in batch]
        texts = [self.processor.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convs]
        imgs, vids = process_vision_info(convs)
        inputs = self.processor(text=texts, images=imgs, videos=vids, padding=True, return_tensors="pt")
        labels = inputs["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        img_token_id = self.processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
        if img_token_id is not None:
            labels[labels == img_token_id] = -100
        inputs["labels"] = labels
        return inputs

collator = ColabDataCollator(processor)

In [ ]:
# [Cell 5] HuggingFace Trainer Execution (Step 3)
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./satquery_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator
)

print("Starting LoRA training on Remote Sensing dataset...")
train_result = trainer.train()

In [ ]:
# [Cell 6] Save & Export LoRA Adapter (Step 4)
EXPORT_DIR = "./exported_models/satquery_qwen2vl_lora"
model.save_pretrained(EXPORT_DIR)
processor.save_pretrained(EXPORT_DIR)
print(f"🎉 Specialist LoRA adapter weights saved successfully to: {EXPORT_DIR}")

In [ ]:
# [Cell 7] Test Production Inference Function (Step 5)
from PIL import Image
import numpy as np

# Create test satellite images
img_t1 = Image.new("RGB", (256, 256), color=(40, 120, 50))
img_t2 = Image.new("RGB", (256, 256), color=(160, 60, 40))

def run_specialist_model(images: list, query: str, task_type: str) -> tuple[str, float]:
    from qwen_vl_utils import process_vision_info
    
    pil_imgs = [im if isinstance(im, Image.Image) else Image.open(im).convert("RGB") for im in images]
    
    user_content = [{"type": "image", "image": im} for im in pil_imgs]
    user_content.append({"type": "text", "text": query})
    conversation = [{"role": "user", "content": user_content}]
    
    text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(conversation)
    
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True
        )
        
    generated_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]
    response = processor.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Confidence calculation
    if outputs.scores:
        probs = torch.softmax(torch.stack(outputs.scores, dim=1), dim=-1)
        token_probs = torch.gather(probs[0], dim=-1, index=generated_ids.unsqueeze(-1)).squeeze(-1)
        conf = float(torch.mean(token_probs).item())
    else:
        conf = 0.95
        
    return response, round(conf, 4)

# Run verification
answer, confidence = run_specialist_model(
    images=[img_t1, img_t2],
    query="What changes occurred between Time 1 and Time 2?",
    task_type="change_detection"
)

print("=" * 60)
print("✨ Inference Verification Output:")
print(f"  • Response   : {answer}")
print(f"  • Confidence : {confidence}")
print("=" * 60)